In [ ]:
import sys
sys.path.append('../src')
from should_be_stdlib import small_then_big_array
from circuit_postprocess import *
from circuits import (
    executor,
    circuit_angle_swap,
    circuit_angle_qft_swap,
    circuit_amp_iamp,
    circuit_amp_iamp_qft,
)
from data import *

In [ ]:
from itertools import combinations_with_replacement

In [ ]:
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
import pandas as pd
import pennylane as qml

In [ ]:
record = load_set()
tuning_curves = get_tc(record)
tuning_curves_rescaled = pd.read_csv(datapath('data_tuning-curves_rescaled.csv'), index_col=0)
tuning_curves_resampled = pd.read_csv(datapath('data_tuning-curves_resampled.csv'), index_col=0)

In [ ]:
dev = qml.device('default.qubit')

In [ ]:
metrics = {
    'ang': (
        tuning_curves_rescaled,
        circuit_angle_swap
    ),
    'ang-qft': (
        tuning_curves_rescaled,
        circuit_angle_qft_swap
    ),
    'amp': (
        tuning_curves_resampled,
        circuit_amp_iamp
    ),
    'amp-qft': (
        tuning_curves_resampled,
        circuit_amp_iamp_qft
    ),
}

## Document & Draw
- calculate circuit types' statistics
- save figures for high-level and once-decomposed circuits

In [ ]:
for (name, (dataset, func)) in metrics.items():
    l = len(dataset.iloc[0])
    a = np.array(list(range(l)), dtype=Float)

    # calculate statistics
    print(f"{name}: {metrics[name][1].__doc__.strip()}")
    print(qml.specs(executor(dev)(metrics[name][1]).original_function, level='device')(a, a)['resources'])
    print()

    # draw abstract circuit
    qml.draw_mpl(executor(dev)(func), level='user', style='pennylane')(a, a)
    plt.suptitle(f"{func.__doc__.strip()}  Circuit", fontsize=20)
    plt.savefig(figspath(f"circuits_{name}.png"), dpi=300)
    plt.close()

    # draw decomposed circuit
    qml.draw_mpl(executor(dev)(func), level='device', style='pennylane')(a, a)
    plt.suptitle(f"{func.__doc__.strip()}  Circuit (Expanded)", fontsize=20)
    plt.savefig(figspath(f"circuits_{name}_expanded.png"), dpi=300)
    plt.close()

## Transpile circuits

- To transpile a pennylane circuit to qasm2 (for qiskit), run it under a quantum tape, then use the qbraid transpiler
- IonQ transpiles to a dictionary which can also be saved for later

In [ ]:
def circuit_to_tape(pl_circuit):
    def tape_machine(*args):
        with qml.tape.QuantumTape() as tape:
            pl_circuit(*args)
        return tape
    return tape_machine

In [ ]:
metric_tape = {
    k: circuit_to_tape(v)
    for (k, (_, v)) in metrics.items()
}

In [ ]:
def get_fidelity_circuit(name_a_b: tuple[str,int,int]):
    name, a, b = name_a_b
    d = metrics[name][0]
    return [
        a, b,
        metric_tape[name](
            *small_then_big_array(
                d.loc[a].to_numpy(),
                d.loc[b].to_numpy()
            )
        ).to_openqasm(measure_all=False)
    ]

In [ ]:
def get_fidelity_circuits(name):
    pairs = combinations_with_replacement(tuning_curves.index, 2)
    pairs_len = len(tuning_curves) * (len(tuning_curves) + 1) // 2

    from multiprocessing import Pool, cpu_count
    with Pool(processes=cpu_count()) as pool:
        ab = list(tqdm(pool.imap(get_fidelity_circuit, [(name,a,b) for (a,b) in pairs]), total=pairs_len, desc=name))

    return pd.DataFrame(ab, columns=['A', 'B', 'qasm2'])

In [ ]:
for (name, (dataset, func)) in metrics.items():
    xlsx = datapath(f"circuits_{name}.xlsx")
    if os.path.exists(xlsx):
        print('circuits already generated for', name)
    else:
        get_fidelity_circuits(name).to_excel(xlsx)